# 04 · Safety stock and days of cover
**Business question:** how much buffer stock does each item need, and is it driven by real demand uncertainty?

**Formula:** Safety stock = Z × σ × √LT
- **Z** = service-level factor, set by ABC class: A = 2.05 (≈98%), B = 1.65 (≈95%), C = 1.28 (≈90%).
- **σ** = weekly standard deviation (from `item_xyz`).
- **LT** = lead time in weeks. **Assumed 2 weeks** here (M5 has no lead-time data).

**Days of cover** = safety stock ÷ average daily demand (= weekly average ÷ 7).

Depends on `item_abc` and `item_xyz` from notebook 03.

> **Run one notebook at a time.** DuckDB allows only one open connection to `practice.duckdb`. Before opening another notebook, use *Kernel → Shut Down Kernel* in this one.

In [ ]:
import duckdb

# Connect to the local DuckDB file built in 00_setup_and_load
con = duckdb.connect('../data/practice.duckdb')

# Memory settings: needed for the 59M-row sales_daily table on a laptop
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## 1. σ of demand vs σ of forecast error
Safety stock should protect against **forecast error**, not raw demand variation. Demand that moves in a predictable way (trend, level shift) is not risk if the forecast follows it.
This compares the two using the naive forecast.

In [ ]:
con.sql("""
WITH weekly_error AS (
    SELECT item_id,
           wm_yr_wk,
           total_units,
           LAG(total_units) OVER (PARTITION BY item_id ORDER BY wm_yr_wk) AS naive_fcst
    FROM item_weekly_units_52w
)
-- CTE needed: STDDEV of a LAG result cannot be written in one step
SELECT item_id,
       ROUND(STDDEV(total_units), 2)              AS std_dev_demand,
       ROUND(STDDEV(naive_fcst - total_units), 2) AS std_dev_error,
       ROUND(STDDEV(naive_fcst - total_units) / STDDEV(total_units), 2) AS ratio_error_to_demand
FROM weekly_error
WHERE naive_fcst IS NOT NULL
GROUP BY item_id
ORDER BY ratio_error_to_demand DESC
""").df()

,item_id,std_dev_demand,std_dev_error,ratio_error_to_demand
0,HOUSEHOLD_2_233,2.80,4.68,1.67
1,HOUSEHOLD_2_017,4.66,7.73,1.66
2,HOUSEHOLD_2_457,6.33,10.48,1.65
3,HOBBIES_1_328,3.37,5.53,1.64
4,FOODS_2_161,4.93,8.11,1.64
...,...,...,...,...
3044,FOODS_3_307,150.50,37.16,0.25
3045,FOODS_3_039,145.86,35.35,0.24
3046,FOODS_3_238,91.45,21.65,0.24
3047,FOODS_3_572,266.58,46.82,0.18


**Finding:** the ratio ranges from 0.18 to 1.67.
- High-volume FOODS items (e.g. FOODS_3_263, ratio 0.18): σ of error is far below σ of demand. Their demand moves, but predictably, so safety stock based on σ of demand would be **too high**.
- Low-volume intermittent items (ratio > 1.6): week-to-week noise makes "last week" a worse guess than the average, so σ of demand would **understate** the real risk for a naive forecast.
- Takeaway: the right σ depends on the forecast in use. The table below uses σ of demand as a simple first version.

## 2. Safety stock by item (largest first)

In [ ]:
con.sql("""
WITH item_z AS (
    SELECT a.item_id,
           a.abc_rev,
           x.avg_weekly_units,
           x.std_dev_units,
           -- service level by value class
           CASE WHEN a.abc_rev = 'A' THEN 2.05
                WHEN a.abc_rev = 'B' THEN 1.65
                ELSE 1.28 END AS z_factor
    FROM item_abc a
    INNER JOIN item_xyz x
            ON a.item_id = x.item_id          -- both item-level, so item_id is the full key
)
-- CTE needed so z_factor can be reused in the formulas below
SELECT item_id,
       abc_rev,
       avg_weekly_units,
       std_dev_units,
       z_factor,
       ROUND(z_factor * std_dev_units * SQRT(2), 0)                        AS safety_stock,   -- LT = 2 weeks
       ROUND(z_factor * std_dev_units * SQRT(2) * 7 / avg_weekly_units, 1) AS doh_coverage    -- days of cover
FROM item_z
ORDER BY safety_stock DESC
""").df()

,item_id,abc_rev,avg_weekly_units,std_dev_units,z_factor,safety_stock,doh_coverage
0,FOODS_3_090,A,3323.403846,1435.075570,2.05,4160.0,8.8
1,FOODS_3_808,A,664.153846,616.038176,2.05,1786.0,18.8
2,FOODS_2_360,A,718.057692,551.288132,2.05,1598.0,15.6
3,FOODS_3_329,A,571.211538,496.762394,2.05,1440.0,17.6
4,FOODS_3_202,A,1146.057692,484.619091,2.05,1405.0,8.6
...,...,...,...,...,...,...,...
3044,HOBBIES_2_016,C,4.615385,2.302206,1.28,4.0,6.3
3045,HOUSEHOLD_2_065,C,4.384615,2.078403,1.28,4.0,6.0
3046,HOUSEHOLD_2_005,C,5.461538,2.388383,1.28,4.0,5.5
3047,HOUSEHOLD_2_276,C,7.307692,2.355956,1.28,4.0,4.1


## 3. Days of cover by item (highest first)

In [ ]:
# Same calculation, sorted by days of cover to find disproportionate buffers
con.sql("""
WITH item_z AS (
    SELECT a.item_id,
           a.abc_rev,
           x.avg_weekly_units,
           x.std_dev_units,
           CASE WHEN a.abc_rev = 'A' THEN 2.05
                WHEN a.abc_rev = 'B' THEN 1.65
                ELSE 1.28 END AS z_factor
    FROM item_abc a
    INNER JOIN item_xyz x
            ON a.item_id = x.item_id
)
SELECT item_id,
       abc_rev,
       avg_weekly_units,
       std_dev_units,
       z_factor,
       ROUND(z_factor * std_dev_units * SQRT(2), 0)                        AS safety_stock,
       ROUND(z_factor * std_dev_units * SQRT(2) * 7 / avg_weekly_units, 1) AS doh_coverage
FROM item_z
ORDER BY doh_coverage DESC
""").df()

,item_id,abc_rev,avg_weekly_units,std_dev_units,z_factor,safety_stock,doh_coverage
0,FOODS_3_441,A,64.750000,172.226581,2.05,499.0,54.0
1,HOBBIES_1_335,C,5.038462,19.131087,1.28,35.0,48.1
2,FOODS_3_444,A,176.826923,398.045505,2.05,1154.0,45.7
3,FOODS_3_069,A,107.326923,230.645022,2.05,669.0,43.6
4,FOODS_2_255,A,130.826923,254.142751,2.05,737.0,39.4
...,...,...,...,...,...,...,...
3044,FOODS_3_458,A,673.653846,63.295722,2.05,184.0,1.9
3045,FOODS_3_697,A,433.288462,40.743741,2.05,118.0,1.9
3046,FOODS_3_108,B,179.250000,19.214858,1.65,45.0,1.8
3047,HOUSEHOLD_1_053,A,203.423077,16.961717,2.05,49.0,1.7


**Findings: safety stock**
- **FOODS_3_090** needs the largest buffer (4,160 units, 8.8 days of cover). Its σ is inflated by weeks of zero sales that look like stockouts (see notebook 05), so part of this buffer is protecting against bad data, not real demand risk.
- **FOODS_3_441** has 54 days of cover: σ (172) is almost 3× its weekly average (65). A buffer this size should trigger a planner review, not be applied automatically.
- Most stable, high-volume items need under 2 days of cover.
- Production version: use σ of forecast error from the real forecast, actual lead times per item, and clean stockout periods out of the history first.